# Online misogyny dataset audit

This notebook inspects the Online Misogyny EACL 2021 dataset stored under `datasets/online-misogyny-eacl2021-main`. It answers the requested audit questions and keeps row-level and unique-post counts separate, because `final_labels.csv` contains repeated `entry_id` values when a post has multiple final labels.

In [11]:
from pathlib import Path
import ast
import hashlib
import os
import re
import time
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)

def find_dataset_root(start=Path.cwd()):
    """Find the dataset whether the notebook runs from the repo root or notebooks/."""
    for directory in [start, *start.parents]:
        candidate = directory / "datasets" / "online-misogyny-eacl2021-main"
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not find datasets/online-misogyny-eacl2021-main from " + str(start))


DATASET_ROOT = find_dataset_root()
PROJECT_ROOT = DATASET_ROOT.parent.parent
DATA_DIR = DATASET_ROOT / "data"
FINAL_CSV = DATA_DIR / "final_labels.csv"
ORIGINAL_CSV = DATA_DIR / "original_labels.csv"
IMAGE_DIR = DATA_DIR / "dataset_post_images"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".gif", ".bmp", ".webp", ".tif", ".tiff"}

for path in [FINAL_CSV, ORIGINAL_CSV, IMAGE_DIR]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Dataset root:", DATASET_ROOT)
print("Final labels:", FINAL_CSV)
print("Image directory:", IMAGE_DIR)

Dataset root: /mnt/course-ee-559/rcp-caas-ee-559-g37/scratch-g37/EE559-DeepLearningProject-g37/datasets/online-misogyny-eacl2021-main
Final labels: /mnt/course-ee-559/rcp-caas-ee-559-g37/scratch-g37/EE559-DeepLearningProject-g37/datasets/online-misogyny-eacl2021-main/data/final_labels.csv
Image directory: /mnt/course-ee-559/rcp-caas-ee-559-g37/scratch-g37/EE559-DeepLearningProject-g37/datasets/online-misogyny-eacl2021-main/data/dataset_post_images


## Load Labels

`keep_default_na=False` keeps empty CSV cells as empty strings. That makes it easier to distinguish missing text from pandas `NaN` values and to inspect unexpected values in the `image` column.

In [12]:
final = pd.read_csv(FINAL_CSV, keep_default_na=False)
original = pd.read_csv(ORIGINAL_CSV, keep_default_na=False)

print(f"final_labels.csv: {len(final):,} rows, {final['entry_id'].nunique():,} unique entry_id values")
print(f"original_labels.csv: {len(original):,} rows, {original['entry_id'].nunique():,} unique entry_id values")
display(final.head(3))

final_labels.csv: 6,567 rows, 6,383 unique entry_id values
original_labels.csv: 15,816 rows, 6,383 unique entry_id values


,entry_id,link_id,parent_id,entry_utc,subreddit,author,body,image,label_date,week,group,sheet_order,level_1,level_2,level_3,strength,highlight,split
0,exoxn7,t3_exoxn7,,1580652620,badwomensanatomy,doggodone,"Do you have the skin of a 80 year old grandma? Worry no more, just drink water!",Yes,17-02-2020,1,1,"(1,)",Nonmisogynistic,None_of_the_categories,,,,train
1,fgb3bdv,t3_exoxn7,t3_exoxn7,1580658139,badwomensanatomy,Machaeon,"This is taking a grain of truth and extrapolating to insanity.\n\nStay hydrated, it's healthy, you'll look and feel better. It will not reverse the aging process though.",,17-02-2020,1,1,"(1, 1)",Nonmisogynistic,None_of_the_categories,,,,train
2,fgc6tlu,t3_exoxn7,t3_exoxn7,1580669695,badwomensanatomy,CuniculusVincitOmnia,Honestly my favorite thing about this is that they feel the need to cite beauty professionals in order to prove that dehydration is caused by not drinking enough water.,,17-02-2020,1,1,"(1, 2)",Nonmisogynistic,None_of_the_categories,,,,test


## Normalize Audit Fields

Some rows contain text fragments in the `image` column instead of the expected blank/`Yes` marker. Those appear to be source CSV quoting issues where an unquoted comma split the post body. The audit keeps the raw column, flags those rows, and uses a repaired `text_for_audit` field for text checks.

In [13]:
def parse_top_sheet_order(value):
    """Return the first integer from sheet_order strings such as '(12, 2, 1)'."""
    try:
        parsed = ast.literal_eval(str(value))
    except (SyntaxError, ValueError):
        return pd.NA
    if isinstance(parsed, tuple) and parsed:
        try:
            return int(parsed[0])
        except (TypeError, ValueError):
            return pd.NA
    return pd.NA


def join_body_fragment(row):
    body = str(row["body_raw"])
    fragment = str(row["image_raw"]).strip()
    if not fragment:
        return body
    return f"{body}, {fragment}" if body.strip() else fragment


df = final.copy()
df["body_raw"] = df["body"].astype(str)
df["image_raw"] = df["image"].astype(str)
df["image_value"] = df["image_raw"].str.strip()
df["image_flag"] = df["image_value"].str.casefold().eq("yes")
df["malformed_image_value"] = (~df["image_flag"]) & df["image_value"].ne("")

df["text_for_audit"] = df["body_raw"]
df.loc[df["malformed_image_value"], "text_for_audit"] = df.loc[df["malformed_image_value"]].apply(join_body_fragment, axis=1)
df["text_empty"] = df["text_for_audit"].str.strip().eq("")

df["sheet_top_order"] = df["sheet_order"].map(parse_top_sheet_order).astype("Int64")
df["expected_image_file"] = ""
image_rows = df["image_flag"] & df["sheet_top_order"].notna()
df.loc[image_rows, "expected_image_file"] = (
    df.loc[image_rows, "week"].astype(str)
    + "_"
    + df.loc[image_rows, "group"].astype(str)
    + "_"
    + df.loc[image_rows, "sheet_top_order"].astype(str)
    + ".jpg"
)
df["expected_image_path"] = df["expected_image_file"].map(lambda name: IMAGE_DIR / name if name else pd.NA)
df["expected_image_exists"] = df["expected_image_file"].map(lambda name: (IMAGE_DIR / name).exists() if name else False)

print("Unexpected non-empty/non-Yes image values:", int(df["malformed_image_value"].sum()))
if df["malformed_image_value"].any():
    display(df.loc[df["malformed_image_value"], ["entry_id", "body", "image", "text_for_audit", "week", "group", "sheet_order"]])

Unexpected non-empty/non-Yes image values: 14


,entry_id,body,image,text_for_audit,week,group,sheet_order
178,fghhw19,You know the red pill is a myth right? The only true pill is the honk pill,everything else is fantasy,"You know the red pill is a myth right? The only true pill is the honk pill, everything else is fantasy",1,2,"(12, 2, 1, 1)"
373,fgw031t,go treat yourself! :) now's a great time to buy jewellery cos of all the promotions anyway,),"go treat yourself! :) now's a great time to buy jewellery cos of all the promotions anyway, )",2,1,"(8, 3, 1, 1, 1, 1)"
673,fid38a2,I don't understand why you'd want to harm your own wife. That's *your* wife,that's like keying your own car.,"I don't understand why you'd want to harm your own wife. That's *your* wife, that's like keying your own car.",3,1,"(1, 10)"
752,fijv8md,I call it BS sindrome,),"I call it BS sindrome, )",3,1,"(7, 4)"
808,fhybtzk,Could be worse,NBC could've sent out Tara and Johnny to cover the game.,"Could be worse, NBC could've sent out Tara and Johnny to cover the game.",3,1,"(11, 3)"
953,fihlafo,Acceptance and being at peace with it,that can be very hard to attain but you are clearly in a very good space and that is a good thing.,"Acceptance and being at peace with it, that can be very hard to attain but you are clearly in a very good space and that is a good thing.",3,2,"(22, 2)"
1321,fizxu7g,I would also love if we get a new jutsu using the jougan,It is tiring to see rasengan and its variations all the time. Kenjutsu it's also cool for Boruto to have and maybe he could make chakra weapons like Momoshiki.,"I would also love if we get a new jutsu using the jougan, It is tiring to see rasengan and its variations all the time. Kenjutsu it's also cool for Boruto to have and maybe he ...",4,1,"(13, 1, 2, 1, 1, 1)"
1613,fj5fi7c,I wish they would fund more Feminist movies,either they'll realize that if they get woke they go broke so they change or Hollywood does go broke and new industries can arise.,"I wish they would fund more Feminist movies, either they'll realize that if they get woke they go broke so they change or Hollywood does go broke and new industries can arise.",4,2,"(36, 2, 1)"
1618,fj5mux2,That's not woman,that's a man. XD,"That's not woman, that's a man. XD",4,2,"(36, 4, 2)"
1867,fek8hm,Anyone here like Dakota R/T's? Pretty sure mine is the cleanest,),"Anyone here like Dakota R/T's? Pretty sure mine is the cleanest, )",5,1,"(20,)"


## Core Dataset Counts

In [14]:
row_label_counts = df["level_1"].value_counts().rename_axis("level_1").reset_index(name="row_count")

def collapse_label(labels):
    values = tuple(sorted(set(labels)))
    if values == ("Misogynistic",):
        return "Misogynistic"
    if values == ("Nonmisogynistic",):
        return "Nonmisogynistic"
    return "Conflict: " + " + ".join(values)

post_summary = (
    df.groupby("entry_id", as_index=True)
    .agg(
        row_count=("entry_id", "size"),
        level_1_collapsed=("level_1", collapse_label),
        has_image=("image_flag", "any"),
        has_empty_text=("text_empty", "any"),
        has_malformed_image_value=("malformed_image_value", "any"),
    )
)
post_label_counts = post_summary["level_1_collapsed"].value_counts().rename_axis("level_1_collapsed").reset_index(name="unique_post_count")

core_counts = pd.DataFrame(
    [
        ("final label rows", len(df)),
        ("unique posts by entry_id", df["entry_id"].nunique()),
        ("misogynistic rows", int((df["level_1"] == "Misogynistic").sum())),
        ("non-misogynistic rows", int((df["level_1"] == "Nonmisogynistic").sum())),
        ("unique misogynistic posts", int((post_summary["level_1_collapsed"] == "Misogynistic").sum())),
        ("unique non-misogynistic posts", int((post_summary["level_1_collapsed"] == "Nonmisogynistic").sum())),
        ("unique posts with conflicting level_1 labels", int(post_summary["level_1_collapsed"].str.startswith("Conflict:").sum())),
        ("image-flagged rows", int(df["image_flag"].sum())),
        ("unique image-flagged posts", int(post_summary["has_image"].sum())),
    ],
    columns=["metric", "value"],
)

display(core_counts)
display(row_label_counts)
display(post_label_counts)

,metric,value
0,final label rows,6567
1,unique posts by entry_id,6383
2,misogynistic rows,699
3,non-misogynistic rows,5868
4,unique misogynistic posts,515
5,unique non-misogynistic posts,5867
6,unique posts with conflicting level_1 labels,1
7,image-flagged rows,173
8,unique image-flagged posts,172


,level_1,row_count
0,Nonmisogynistic,5868
1,Misogynistic,699


,level_1_collapsed,unique_post_count
0,Nonmisogynistic,5867
1,Misogynistic,515
2,Conflict: Misogynistic + Nonmisogynistic,1


## Unicode Emoji Check

This counts Unicode emoji code points in the raw/repaired post text. ASCII emoticons such as `:)` are not counted as Unicode emoji.

In [15]:
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F1E6-\U0001F1FF"
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FA6F"
    "\U0001FA70-\U0001FAFF"
    "\u2600-\u27BF"
    "]"
)

OCR_EMOTICON_PATTERN = re.compile(
    r"""(?ix)
    (?<![A-Za-z0-9_])
    (
        <3
        | [xX]-?[dDpP]
        | [:;=8]["'`\-oO]?[)D(Pp/\\|sS3]
        | \^[._-]?\^
        | :[a-z][a-z0-9_+\-]*:
    )
    (?![A-Za-z0-9_])
    """
)
OCR_LAUGHTER_PATTERN = re.compile(r"(?i)\b(?:lol|lmao|rofl)\b")


def find_unicode_emojis(text):
    return [match.group(0) for match in EMOJI_PATTERN.finditer(str(text))]


def find_ocr_emoticons(text):
    return [match.group(0) for match in OCR_EMOTICON_PATTERN.finditer(str(text))]


def find_ocr_laughter_tokens(text):
    return [match.group(0) for match in OCR_LAUGHTER_PATTERN.finditer(str(text))]


def emoji_tokens_to_text(tokens):
    return " ".join(str(token) for token in tokens if str(token).strip())


df["has_unicode_emoji"] = df["text_for_audit"].map(lambda text: bool(find_unicode_emojis(text)))
emoji_rows = int(df["has_unicode_emoji"].sum())
emoji_posts = int(df.loc[df["has_unicode_emoji"], "entry_id"].nunique())

print(f"Rows containing Unicode emoji: {emoji_rows:,}")
print(f"Unique posts containing Unicode emoji: {emoji_posts:,}")

if emoji_rows:
    display(df.loc[df["has_unicode_emoji"], ["entry_id", "text_for_audit", "level_1"]].head(20))


Rows containing Unicode emoji: 0
Unique posts containing Unicode emoji: 0


## Image File Checks

Image filenames appear to follow `{week}_{group}_{top_level_sheet_order}.jpg` for rows where `image == Yes`. The audit uses that convention to check for missing or extra files, while also reporting the raw number of image files found on disk.

In [16]:
image_files = sorted([p for p in IMAGE_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])
non_image_files = sorted([p for p in IMAGE_DIR.iterdir() if p.is_file() and p.suffix.lower() not in IMAGE_EXTENSIONS])
actual_image_names = {p.name for p in image_files}
expected_image_names = set(df.loc[df["image_flag"], "expected_image_file"]) - {""}
missing_expected_images = sorted(expected_image_names - actual_image_names)
extra_image_files = sorted(actual_image_names - expected_image_names)

image_inspection = []
for path in image_files:
    row = {"file": path.name, "path": str(path), "readable": True, "width": pd.NA, "height": pd.NA, "error": ""}
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            row["width"], row["height"] = img.size
    except Exception as exc:
        row["readable"] = False
        row["error"] = repr(exc)
    image_inspection.append(row)

image_inspection_df = pd.DataFrame(image_inspection)
broken_images = image_inspection_df.loc[~image_inspection_df["readable"]] if len(image_inspection_df) else pd.DataFrame()

hash_groups = defaultdict(list)
for path in image_files:
    if path.exists() and path.is_file():
        hash_groups[hashlib.md5(path.read_bytes()).hexdigest()].append(path.name)
duplicate_image_groups = [sorted(names) for names in hash_groups.values() if len(names) > 1]

image_quality_summary = pd.DataFrame(
    [
        ("image-flagged rows", int(df["image_flag"].sum())),
        ("unique image-flagged posts", int(df.loc[df["image_flag"], "entry_id"].nunique())),
        ("unique expected image filenames", len(expected_image_names)),
        ("image files found on disk", len(image_files)),
        ("non-image files in image directory", len(non_image_files)),
        ("missing expected image files", len(missing_expected_images)),
        ("extra image files not mapped from labels", len(extra_image_files)),
        ("broken/unreadable image files", int((~image_inspection_df["readable"]).sum()) if len(image_inspection_df) else 0),
        ("duplicate image-content groups", len(duplicate_image_groups)),
        ("extra duplicate image files by content", sum(len(group) - 1 for group in duplicate_image_groups)),
    ],
    columns=["metric", "value"],
)

display(image_quality_summary)

print("Missing expected image files:")
display(pd.DataFrame({"missing_expected_image_file": missing_expected_images}))

print("Extra image files not mapped from labels:")
display(pd.DataFrame({"extra_image_file": extra_image_files}))

print("Non-image files in image directory:")
display(pd.DataFrame({"non_image_file": [p.name for p in non_image_files]}))

print("Duplicate image content groups:")
display(pd.DataFrame({"duplicate_group": duplicate_image_groups}))

if len(broken_images):
    display(broken_images)

,metric,value
0,image-flagged rows,173
1,unique image-flagged posts,172
2,unique expected image filenames,172
3,image files found on disk,169
4,non-image files in image directory,1
5,missing expected image files,29
6,extra image files not mapped from labels,26
7,broken/unreadable image files,0
8,duplicate image-content groups,2
9,extra duplicate image files by content,2


Missing expected image files:


,missing_expected_image_file
0,10_2_49.jpg
1,10_2_57.jpg
2,10_2_63.jpg
3,10_2_64.jpg
4,10_2_69.jpg
5,10_2_72.jpg
6,10_2_75.jpg
7,10_2_78.jpg
8,10_2_84.jpg
9,1_1_1.jpg


Extra image files not mapped from labels:


,extra_image_file
0,10_2_50.jpg
1,10_2_59.jpg
2,10_2_65.jpg
3,10_2_66.jpg
4,10_2_73.jpg
5,10_2_74.jpg
6,10_2_77.jpg
7,10_2_82.jpg
8,10_2_88.jpg
9,11_2_78.jpg


Non-image files in image directory:


,non_image_file
0,.DS_Store


Duplicate image content groups:


,duplicate_group
0,"[8_1_7.jpg, 9_2_55.jpg]"
1,"[9_1_2.jpg, 9_2_58.jpg]"


## EasyOCR: Readable Text Inside Images

From this point onward the notebook uses EasyOCR only. The OCR pass reads each image once, normalizes whitespace, and leaves `ocr_text` empty for posts without image text or when OCR cannot be computed. Set `RUN_EASYOCR=0` before launching the notebook if you need to skip the slow OCR pass during a quick audit run.


In [17]:
MIN_READABLE_CHARS = 3
EASYOCR_LANGUAGES = ["en"]
EASYOCR_USE_GPU = False
RUN_EASYOCR = os.environ.get("RUN_EASYOCR", "1") != "0"
REFRESH_OCR_CACHE = os.environ.get("REFRESH_OCR_CACHE", "0") == "1"
EASYOCR_CACHE_DIR = Path(os.environ.get("EASYOCR_CACHE_DIR", PROJECT_ROOT / ".easyocr"))
EASYOCR_MODEL_DIR = EASYOCR_CACHE_DIR / "model"
EASYOCR_USER_NETWORK_DIR = EASYOCR_CACHE_DIR / "user_network"
OCR_RESULTS_CACHE = Path(os.environ.get("OCR_RESULTS_CACHE", EASYOCR_CACHE_DIR / "ocr_results.csv"))
OCR_ENGINE_NAME = "easyocr"
OCR_LANGUAGES_KEY = ",".join(EASYOCR_LANGUAGES)
OCR_CACHE_COLUMNS = [
    "image_id",
    "file",
    "image_path",
    "image_size_bytes",
    "image_mtime_ns",
    "ocr_engine",
    "ocr_engine_version",
    "ocr_languages",
    "ocr_text",
    "ocr_text_clean",
    "ocr_success",
    "ocr_status",
    "has_readable_text",
    "unicode_emojis_ocr_text",
    "emoticons_ocr_text",
    "laughter_tokens_ocr_text",
    "ocr_confidence_mean",
    "ocr_confidence_min",
    "ocr_detection_count",
    "processing_time_seconds",
    "error",
    "cached_at",
]
OCR_NUMERIC_COLUMNS = [
    "image_size_bytes",
    "image_mtime_ns",
    "ocr_confidence_mean",
    "ocr_confidence_min",
    "ocr_detection_count",
    "processing_time_seconds",
]
OCR_BOOL_COLUMNS = ["ocr_success", "has_readable_text"]

if RUN_EASYOCR:
    try:
        import easyocr
    except ImportError as exc:
        easyocr = None
        easyocr_import_error = exc
    else:
        easyocr_import_error = None
else:
    easyocr = None
    easyocr_import_error = None

OCR_ENGINE_VERSION = getattr(easyocr, "__version__", "") if easyocr is not None else ""


def normalize_ocr_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def project_relative_path(path):
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def image_metadata(path):
    stat = path.stat()
    return {
        "image_id": path.stem,
        "file": path.name,
        "image_path": project_relative_path(path),
        "image_size_bytes": int(stat.st_size),
        "image_mtime_ns": int(stat.st_mtime_ns),
        "ocr_engine": OCR_ENGINE_NAME,
        "ocr_engine_version": OCR_ENGINE_VERSION,
        "ocr_languages": OCR_LANGUAGES_KEY,
    }


def parse_bool(value):
    if isinstance(value, bool):
        return value
    return str(value).strip().casefold() in {"true", "1", "yes", "y"}


def safe_int(value, default=-1):
    try:
        if pd.isna(value):
            return default
        return int(value)
    except (TypeError, ValueError):
        return default


def add_ocr_token_columns(results_df):
    if len(results_df) == 0:
        return results_df
    results_df = results_df.copy()
    text = results_df["ocr_text_clean"].fillna("")
    results_df["unicode_emojis_ocr_text"] = text.map(lambda value: emoji_tokens_to_text(find_unicode_emojis(value)))
    results_df["emoticons_ocr_text"] = text.map(lambda value: emoji_tokens_to_text(find_ocr_emoticons(value)))
    results_df["laughter_tokens_ocr_text"] = text.map(lambda value: emoji_tokens_to_text(find_ocr_laughter_tokens(value)))
    return results_df


def read_ocr_cache(cache_path):
    if not cache_path.exists():
        return pd.DataFrame(columns=OCR_CACHE_COLUMNS)

    cache = pd.read_csv(cache_path, keep_default_na=False)
    for column in OCR_CACHE_COLUMNS:
        if column not in cache.columns:
            cache[column] = ""
    for column in OCR_NUMERIC_COLUMNS:
        cache[column] = pd.to_numeric(cache[column], errors="coerce")
    for column in OCR_BOOL_COLUMNS:
        cache[column] = cache[column].map(parse_bool)
    cache = add_ocr_token_columns(cache)
    return cache[OCR_CACHE_COLUMNS]


def cached_row_for_image(path, cache_df):
    if REFRESH_OCR_CACHE or len(cache_df) == 0:
        return None

    metadata = image_metadata(path)
    candidates = cache_df.loc[cache_df["file"].eq(metadata["file"])]
    for _, cached in candidates.iloc[::-1].iterrows():
        size_matches = safe_int(cached.get("image_size_bytes", -1)) == metadata["image_size_bytes"]
        mtime_matches = safe_int(cached.get("image_mtime_ns", -1)) == metadata["image_mtime_ns"]
        engine_matches = str(cached.get("ocr_engine", "")) == OCR_ENGINE_NAME
        language_matches = str(cached.get("ocr_languages", "")) == OCR_LANGUAGES_KEY
        if size_matches and mtime_matches and engine_matches and language_matches:
            row = {column: cached.get(column, "") for column in OCR_CACHE_COLUMNS}
            row["cache_hit"] = True
            return row
    return None


def empty_ocr_row(path, status, error=""):
    row = image_metadata(path)
    row.update(
        {
            "ocr_text": "",
            "ocr_text_clean": "",
            "ocr_success": False,
            "ocr_status": status,
            "has_readable_text": False,
            "unicode_emojis_ocr_text": "",
            "emoticons_ocr_text": "",
            "laughter_tokens_ocr_text": "",
            "ocr_confidence_mean": np.nan,
            "ocr_confidence_min": np.nan,
            "ocr_detection_count": 0,
            "processing_time_seconds": 0.0,
            "error": error,
            "cached_at": pd.Timestamp.utcnow().isoformat(),
            "cache_hit": False,
        }
    )
    return row


def read_image_with_easyocr(path, reader):
    detections = reader.readtext(str(path), detail=1, paragraph=False)
    pieces = []
    confidences = []
    for detection in detections:
        if len(detection) >= 2 and str(detection[1]).strip():
            pieces.append(str(detection[1]).strip())
        if len(detection) >= 3:
            try:
                confidences.append(float(detection[2]))
            except (TypeError, ValueError):
                pass
    confidence_mean = float(np.mean(confidences)) if confidences else np.nan
    confidence_min = float(np.min(confidences)) if confidences else np.nan
    return "\n".join(pieces), confidence_mean, confidence_min, len(detections)


def run_easyocr_for_image(path, reader):
    start_time = time.perf_counter()
    try:
        text, confidence_mean, confidence_min, detection_count = read_image_with_easyocr(path, reader)
        error = ""
        ocr_success = True
    except Exception as exc:
        text = ""
        confidence_mean = np.nan
        confidence_min = np.nan
        detection_count = 0
        error = repr(exc)
        ocr_success = False

    elapsed = time.perf_counter() - start_time
    cleaned = normalize_ocr_text(text)
    has_readable_text = bool(re.search(r"[A-Za-z0-9]", cleaned)) and len(cleaned) >= MIN_READABLE_CHARS
    row = image_metadata(path)
    row.update(
        {
            "ocr_text": text,
            "ocr_text_clean": cleaned,
            "ocr_success": ocr_success,
            "ocr_status": "success" if ocr_success else "failure",
            "has_readable_text": has_readable_text,
            "unicode_emojis_ocr_text": emoji_tokens_to_text(find_unicode_emojis(cleaned)),
            "emoticons_ocr_text": emoji_tokens_to_text(find_ocr_emoticons(cleaned)),
            "laughter_tokens_ocr_text": emoji_tokens_to_text(find_ocr_laughter_tokens(cleaned)),
            "ocr_confidence_mean": confidence_mean,
            "ocr_confidence_min": confidence_min,
            "ocr_detection_count": detection_count,
            "processing_time_seconds": elapsed,
            "error": error,
            "cached_at": pd.Timestamp.utcnow().isoformat(),
            "cache_hit": False,
        }
    )
    return row


def write_ocr_cache(results_df, cache_path):
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_df = add_ocr_token_columns(results_df.drop(columns=["cache_hit"], errors="ignore"))
    cache_df[OCR_CACHE_COLUMNS].to_csv(cache_path, index=False)


ocr_status = "not computed"
readable_image_file_count = None
readable_image_post_count = None
ocr_cache = read_ocr_cache(OCR_RESULTS_CACHE)
ocr_rows = [None] * len(image_files)
pending_images = []
cache_hits = 0
computed_images = 0
failed_images = 0

for index, path in enumerate(image_files):
    cached = cached_row_for_image(path, ocr_cache)
    if cached is None:
        pending_images.append((index, path))
    else:
        ocr_rows[index] = cached
        cache_hits += 1

if len(image_files) == 0:
    ocr_status = "not computed: no image files found"
elif not RUN_EASYOCR:
    for index, path in pending_images:
        ocr_rows[index] = empty_ocr_row(path, "skipped: RUN_EASYOCR=0")
    ocr_status = f"loaded OCR cache only ({cache_hits:,} cache hits, {len(pending_images):,} missing)"
elif easyocr is None:
    for index, path in pending_images:
        ocr_rows[index] = empty_ocr_row(path, "skipped: EasyOCR is not installed", repr(easyocr_import_error))
    ocr_status = f"loaded OCR cache only ({cache_hits:,} cache hits); EasyOCR is not installed"
    print("Install EasyOCR in the active environment with `python -m pip install easyocr`.")
elif len(pending_images) == 0:
    ocr_status = f"loaded from OCR cache ({cache_hits:,} cache hits); cache: {OCR_RESULTS_CACHE}"
else:
    try:
        EASYOCR_MODEL_DIR.mkdir(parents=True, exist_ok=True)
        EASYOCR_USER_NETWORK_DIR.mkdir(parents=True, exist_ok=True)
        easyocr_reader = easyocr.Reader(
            EASYOCR_LANGUAGES,
            gpu=EASYOCR_USE_GPU,
            model_storage_directory=str(EASYOCR_MODEL_DIR),
            user_network_directory=str(EASYOCR_USER_NETWORK_DIR),
            verbose=False,
        )
    except Exception as exc:
        setup_error = repr(exc)
        for index, path in pending_images:
            ocr_rows[index] = empty_ocr_row(path, "skipped: EasyOCR setup failed", setup_error)
        ocr_status = f"loaded OCR cache only ({cache_hits:,} cache hits); EasyOCR setup failed"
    else:
        for index, path in pending_images:
            row = run_easyocr_for_image(path, easyocr_reader)
            ocr_rows[index] = row
            computed_images += 1
            failed_images += int(not row["ocr_success"])
        ocr_status = (
            f"computed with EasyOCR ({computed_images:,} processed, {cache_hits:,} cache hits, "
            f"{failed_images:,} failures); cache: {OCR_RESULTS_CACHE}"
        )

ocr_results = pd.DataFrame([row for row in ocr_rows if row is not None])
for column in OCR_BOOL_COLUMNS:
    if column in ocr_results.columns:
        ocr_results[column] = ocr_results[column].map(parse_bool)
ocr_results = add_ocr_token_columns(ocr_results)

if RUN_EASYOCR and easyocr is not None and computed_images > 0:
    write_ocr_cache(ocr_results, OCR_RESULTS_CACHE)

readable_files = set(ocr_results.loc[ocr_results["has_readable_text"], "file"]) if len(ocr_results) else set()
readable_image_file_count = len(readable_files)
readable_image_post_count = df.loc[df["expected_image_file"].isin(readable_files), "entry_id"].nunique()
ocr_success_count = int(ocr_results["ocr_success"].sum()) if len(ocr_results) else 0
ocr_failure_count = int((~ocr_results["ocr_success"]).sum()) if len(ocr_results) else 0

ocr_cache_summary = pd.DataFrame(
    [
        ("image files considered", len(image_files)),
        ("OCR cache hits", cache_hits),
        ("images processed this run", computed_images),
        ("OCR successes", ocr_success_count),
        ("OCR failures/skips", ocr_failure_count),
        ("image files with readable OCR text", readable_image_file_count),
        ("mapped unique image posts with readable OCR text", readable_image_post_count),
    ],
    columns=["metric", "value"],
)

print("OCR status:", ocr_status)
print("OCR cache:", OCR_RESULTS_CACHE)
display(ocr_cache_summary)
display(
    ocr_results[
        [
            "image_id",
            "file",
            "ocr_success",
            "ocr_status",
            "ocr_confidence_mean",
            "processing_time_seconds",
            "cache_hit",
            "ocr_text_clean",
        ]
    ].head(30)
)


OCR status: loaded from OCR cache (169 cache hits); cache: /mnt/course-ee-559/rcp-caas-ee-559-g37/scratch-g37/EE559-DeepLearningProject-g37/.easyocr/ocr_results.csv
OCR cache: /mnt/course-ee-559/rcp-caas-ee-559-g37/scratch-g37/EE559-DeepLearningProject-g37/.easyocr/ocr_results.csv


,metric,value
0,image files considered,169
1,OCR cache hits,169
2,images processed this run,0
3,OCR successes,169
4,OCR failures/skips,0
5,image files with readable OCR text,146
6,mapped unique image posts with readable OCR text,125


,image_id,file,ocr_success,ocr_status,ocr_confidence_mean,processing_time_seconds,cache_hit,ocr_text_clean
0,10_1_1,10_1_1.jpg,True,success,0.840256,1.062802,True,Onyekachi @its_roi Can sex doll cook your meals? Can sex dollhave your kids? Can sex doll wash your clothes? Can sex doll make your home? What's it? You'Il are really triggered...
1,10_1_12,10_1_12.jpg,True,success,0.700067,3.956168,True,"Bois Locker Room: Girl created fake account to suggest sex assault plan Times News Network sault, apparently t0 see how he AVIRALTALE reacted: New Delhl: It wasagirdwhohad The ..."
2,10_1_13,10_1_13.jpg,True,success,0.702590,4.832044,True,"""typet The sex Give it"
3,10_1_19,10_1_19.jpg,True,success,NaN,0.187478,True,
4,10_1_3,10_1_3.jpg,True,success,0.343970,0.905308,True,Dolile BLONSKY ET 4L _ Gue APTARATUS FO)R Facilitating THE WkTm CHMD CNTRIFUGAL ORCEA
5,10_1_32,10_1_32.jpg,True,success,0.863987,0.489900,True,jeanluc is online only @jeanluc 3 being ugly is so disheartening you be having a good time and then glimpse of ya reflection and it kill ur whole mood get
6,10_1_33,10_1_33.jpg,True,success,0.831730,1.549119,True,"A Video From 3 Years Resurfaced Of Cardi B Admitting To Drugging And Robhing Men After Thinking Were Going To Have Sex: Women Are Going Hard For Her Saying ""It was years #g0 an..."
7,10_1_35,10_1_35.jpg,True,success,0.823012,1.798645,True,"Anonymous 09/26/19(Thu)21.58.59 No.121597357 >Goes to prostitute >""Alright honey what do ya want? $50 to blow $100 for the night. $250 for anal, but ya gotta wear a 40 KB JPG r..."
8,10_1_36,10_1_36.jpg,True,success,0.913527,7.237655,True,Queens Leveling Up LEVEL UP r/ FemaleDatingStrategy FDS
9,10_1_37,10_1_37.jpg,True,success,0.348486,3.218678,True,"told my sister to write an essay on the wage gap for her class & this is her professor's reply Hi Jane First 0#, youf premise iS Wtong The wage gap IS vory real: So the reason ..."


## One Row Per Post

This builds `post_df`, a single dataframe with one row per post/sample. It keeps the required raw-text, OCR-text, image, label, emoji, OCR-confidence, failure, and notes fields, while retaining the OCR emoji-proxy fields used earlier.


In [18]:
RAW_TEXT_SOURCE_COLUMNS = [column for column in ["title", "caption", "body"] if column in df.columns]
POST_DF_MINIMUM_COLUMNS = [
    "sample_id",
    "label",
    "is_image_post",
    "image_path",
    "raw_text",
    "ocr_text",
    "raw_text_len",
    "ocr_text_len",
    "n_unicode_emojis_raw",
    "emoji_list_raw",
    "n_unicode_emojis_ocr",
    "emoji_list_ocr",
    "has_any_emoji",
    "has_ocr_text",
    "ocr_confidence_mean",
    "ocr_failed",
    "notes",
]


def unique_nonempty_values(values):
    unique = []
    seen = set()
    for value in values:
        if value is pd.NA:
            continue
        text = str(value).strip()
        if not text or text in seen:
            continue
        unique.append(text)
        seen.add(text)
    return unique


def merge_unique_nonempty_text(values):
    return "\n".join(unique_nonempty_values(values))


def raw_text_from_row(row):
    pieces = []
    for column in RAW_TEXT_SOURCE_COLUMNS:
        value = str(row.get(column, "")).strip()
        if value:
            pieces.append(value)

    # Some source CSV rows split part of the body into the image column. Keep that recovered text.
    if bool(row.get("malformed_image_value", False)):
        image_fragment = str(row.get("image_raw", "")).strip()
        if image_fragment and image_fragment.casefold() != "yes":
            pieces.append(image_fragment)

    if not pieces:
        fallback = str(row.get("text_for_audit", "")).strip()
        if fallback:
            pieces.append(fallback)

    return merge_unique_nonempty_text(pieces)


def find_ocr_emoji_proxy_tokens(text):
    text = str(text)
    tokens = []
    tokens.extend(find_unicode_emojis(text))
    tokens.extend(find_ocr_emoticons(text))
    tokens.extend(find_ocr_laughter_tokens(text))
    return tokens


def image_emoji_heuristic_tokens(path):
    """Lightweight proxy flags for graphical emoji-like marks; these are cues, not labels."""
    try:
        with Image.open(path) as img:
            img = img.convert("RGB")
            img.thumbnail((256, 256))
            pixels = np.asarray(img).astype(np.float32)
    except Exception:
        return []

    if pixels.size == 0:
        return []

    red = pixels[:, :, 0]
    green = pixels[:, :, 1]
    blue = pixels[:, :, 2]
    saturation = pixels.max(axis=2) - pixels.min(axis=2)

    yellow_orange = (red > 150) & (green > 100) & (blue < 130) & (saturation > 40) & ((red - green) < 100)
    red_pink = (red > 150) & (green < 120) & (blue < 160) & (saturation > 50)

    tokens = []
    yellow_ratio = float(yellow_orange.mean())
    red_pink_ratio = float(red_pink.mean())
    if 0.001 <= yellow_ratio <= 0.25:
        tokens.append("possible_yellow_emoji_graphic")
    if 0.0005 <= red_pink_ratio <= 0.15:
        tokens.append("possible_red_heart_or_symbol")
    return tokens


def unique_token_text(tokens):
    unique = []
    seen = set()
    for token in tokens:
        token = str(token).strip()
        if not token or token in seen:
            continue
        unique.append(token)
        seen.add(token)
    return " ".join(unique)


def image_paths_for_files(files):
    return "\n".join(project_relative_path(IMAGE_DIR / file_name) for file_name in files if file_name)


def missing_image_files(files):
    return [file_name for file_name in files if file_name and not (IMAGE_DIR / file_name).exists()]


def ocr_text_for_files(files):
    pieces = []
    for file_name in files:
        row = ocr_row_by_file.get(file_name)
        if row is not None and parse_bool(row.get("ocr_success", False)):
            text = str(row.get("ocr_text_clean", "")).strip()
            if text:
                pieces.append(text)
    return "\n".join(pieces)


def ocr_confidence_mean_for_files(files):
    values = []
    for file_name in files:
        row = ocr_row_by_file.get(file_name)
        if row is None or not parse_bool(row.get("ocr_success", False)):
            continue
        value = pd.to_numeric(row.get("ocr_confidence_mean", np.nan), errors="coerce")
        if not pd.isna(value):
            values.append(float(value))
    return float(np.mean(values)) if values else np.nan


def ocr_failed_for_files(files):
    if not files:
        return False
    for file_name in files:
        if not (IMAGE_DIR / file_name).exists():
            return True
        row = ocr_row_by_file.get(file_name)
        if row is None or not parse_bool(row.get("ocr_success", False)):
            return True
    return False


def ocr_statuses_for_files(files):
    statuses = []
    for file_name in files:
        row = ocr_row_by_file.get(file_name)
        if row is None:
            statuses.append(f"{file_name}:missing_ocr_result")
        else:
            statuses.append(f"{file_name}:{row.get('ocr_status', '')}")
    return statuses


def notes_for_post(row):
    notes = []
    if not str(row["raw_text"]).strip():
        notes.append("empty_raw_text")
    if str(row["label"]).startswith("Conflict:"):
        notes.append("label_conflict")
    if bool(row["has_malformed_image_value"]):
        notes.append("raw_text_repaired_from_image_column")
    if row["is_image_post"] and not row["expected_image_files"]:
        notes.append("image_flag_without_expected_filename")
    if row["missing_image_files"]:
        notes.append("missing_image_file=" + ",".join(row["missing_image_files"]))
    if row["is_image_post"] and row["ocr_failed"]:
        notes.append("ocr_failed=" + ",".join(ocr_statuses_for_files(row["expected_image_files"])))
    if row["is_image_post"] and not row["has_ocr_text"] and not row["ocr_failed"]:
        notes.append("no_readable_ocr_text")
    return "; ".join(notes)


if len(ocr_results):
    ocr_results["unicode_emojis_ocr_text"] = ocr_results["ocr_text_clean"].map(lambda text: emoji_tokens_to_text(find_unicode_emojis(text)))
    ocr_results["emoticons_ocr_text"] = ocr_results["ocr_text_clean"].map(lambda text: emoji_tokens_to_text(find_ocr_emoticons(text)))
    ocr_results["laughter_tokens_ocr_text"] = ocr_results["ocr_text_clean"].map(lambda text: emoji_tokens_to_text(find_ocr_laughter_tokens(text)))

ocr_emoji_summary = pd.DataFrame(
    [
        ("images with Unicode emoji in OCR text", int(ocr_results["unicode_emojis_ocr_text"].str.strip().ne("").sum()) if len(ocr_results) else 0),
        ("images with emoticons in OCR text", int(ocr_results["emoticons_ocr_text"].str.strip().ne("").sum()) if len(ocr_results) else 0),
        ("images with laughter tokens in OCR text", int(ocr_results["laughter_tokens_ocr_text"].str.strip().ne("").sum()) if len(ocr_results) else 0),
    ],
    columns=["metric", "value"],
)

ocr_row_by_file = ocr_results.set_index("file").to_dict(orient="index") if len(ocr_results) else {}
image_path_by_file = {path.name: path for path in image_files}
image_heuristic_tokens_by_file = {
    file_name: image_emoji_heuristic_tokens(path)
    for file_name, path in image_path_by_file.items()
}

raw_text_candidates = df.apply(raw_text_from_row, axis=1)
post_df = (
    df.assign(_raw_text_candidate=raw_text_candidates)
    .groupby("entry_id", sort=False)
    .agg(
        raw_text=("_raw_text_candidate", merge_unique_nonempty_text),
        is_image_post=("image_flag", "any"),
        expected_image_files=("expected_image_file", unique_nonempty_values),
        has_malformed_image_value=("malformed_image_value", "any"),
    )
    .reset_index()
    .rename(columns={"entry_id": "sample_id"})
)

post_df["label"] = post_df["sample_id"].map(post_summary["level_1_collapsed"])
post_df["image_path"] = post_df["expected_image_files"].map(image_paths_for_files)
post_df["missing_image_files"] = post_df["expected_image_files"].map(missing_image_files)
post_df["ocr_text"] = post_df["expected_image_files"].map(ocr_text_for_files)
post_df["raw_text_len"] = post_df["raw_text"].map(lambda text: len(str(text)))
post_df["ocr_text_len"] = post_df["ocr_text"].map(lambda text: len(str(text)))
post_df["emoji_list_raw"] = post_df["raw_text"].map(find_unicode_emojis)
post_df["n_unicode_emojis_raw"] = post_df["emoji_list_raw"].map(len)
post_df["emoji_list_ocr"] = post_df["ocr_text"].map(find_unicode_emojis)
post_df["n_unicode_emojis_ocr"] = post_df["emoji_list_ocr"].map(len)
post_df["ocr_emoticons"] = post_df["ocr_text"].map(lambda text: emoji_tokens_to_text(find_ocr_emoticons(text)))
post_df["ocr_laughter_tokens"] = post_df["ocr_text"].map(lambda text: emoji_tokens_to_text(find_ocr_laughter_tokens(text)))
post_df["has_ocr_text"] = post_df["ocr_text"].str.strip().ne("")
post_df["ocr_confidence_mean"] = post_df["expected_image_files"].map(ocr_confidence_mean_for_files)
post_df["ocr_failed"] = post_df["expected_image_files"].map(ocr_failed_for_files)


def image_proxy_for_row(row):
    tokens = find_ocr_emoji_proxy_tokens(row["ocr_text"])
    for file_name in row["expected_image_files"]:
        tokens.extend(image_heuristic_tokens_by_file.get(file_name, []))
    return unique_token_text(tokens)


post_df["emoji_in_image_proxy"] = post_df.apply(image_proxy_for_row, axis=1)
post_df["unicode_emojis_text"] = post_df["emoji_list_raw"].map(emoji_tokens_to_text)
post_df["has_any_emoji"] = (
    post_df["n_unicode_emojis_raw"].gt(0)
    | post_df["n_unicode_emojis_ocr"].gt(0)
    | post_df["ocr_emoticons"].str.strip().ne("")
    | post_df["ocr_laughter_tokens"].str.strip().ne("")
    | post_df["emoji_in_image_proxy"].str.strip().ne("")
)
post_df["notes"] = post_df.apply(notes_for_post, axis=1)

extra_post_columns = [
    "expected_image_files",
    "missing_image_files",
    "unicode_emojis_text",
    "ocr_emoticons",
    "ocr_laughter_tokens",
    "emoji_in_image_proxy",
]
post_df = post_df[POST_DF_MINIMUM_COLUMNS + extra_post_columns]
post_text_features = post_df

row_level_post_fields = post_df[
    ["sample_id", "raw_text", "ocr_text", "unicode_emojis_text", "emoji_in_image_proxy"]
].rename(columns={"sample_id": "entry_id"})
feature_columns = ["raw_text", "ocr_text", "unicode_emojis_text", "emoji_in_image_proxy"]
df = df.drop(columns=[column for column in feature_columns if column in df.columns], errors="ignore").merge(
    row_level_post_fields,
    on="entry_id",
    how="left",
)

post_df_summary = pd.DataFrame(
    [
        ("post_df rows", len(post_df)),
        ("unique sample_id values", post_df["sample_id"].nunique()),
        ("image posts", int(post_df["is_image_post"].sum())),
        ("posts with OCR text", int(post_df["has_ocr_text"].sum())),
        ("posts with OCR failure", int(post_df["ocr_failed"].sum())),
        ("posts with any emoji/proxy token", int(post_df["has_any_emoji"].sum())),
    ],
    columns=["metric", "value"],
)

display(ocr_emoji_summary)
display(post_df_summary)
display(post_df[POST_DF_MINIMUM_COLUMNS].head(20))


,metric,value
0,images with Unicode emoji in OCR text,0
1,images with emoticons in OCR text,0
2,images with laughter tokens in OCR text,10


,metric,value
0,post_df rows,6383
1,unique sample_id values,6383
2,image posts,172
3,posts with OCR text,125
4,posts with OCR failure,29
5,posts with any emoji/proxy token,109


,sample_id,label,is_image_post,image_path,raw_text,ocr_text,raw_text_len,ocr_text_len,n_unicode_emojis_raw,emoji_list_raw,n_unicode_emojis_ocr,emoji_list_ocr,has_any_emoji,has_ocr_text,ocr_confidence_mean,ocr_failed,notes
0,exoxn7,Nonmisogynistic,True,datasets/online-misogyny-eacl2021-main/data/dataset_post_images/1_1_1.jpg,"Do you have the skin of a 80 year old grandma? Worry no more, just drink water!",,79,0,0,[],0,[],False,False,NaN,True,missing_image_file=1_1_1.jpg; ocr_failed=1_1_1.jpg:missing_ocr_result
1,fgb3bdv,Nonmisogynistic,False,,"This is taking a grain of truth and extrapolating to insanity.\n\nStay hydrated, it's healthy, you'll look and feel better. It will not reverse the aging process though.",,167,0,0,[],0,[],False,False,NaN,False,
2,fgc6tlu,Nonmisogynistic,False,,Honestly my favorite thing about this is that they feel the need to cite beauty professionals in order to prove that dehydration is caused by not drinking enough water.,,168,0,0,[],0,[],False,False,NaN,False,
3,fge6msg,Nonmisogynistic,False,,Source? Doesnt sound right to me idk,,36,0,0,[],0,[],False,False,NaN,False,
4,fgawus5,Misogynistic,False,,"Damn, I saw a movie in which the old woman bathed in the blood if virgins to do this. How did no one tell her she just needed some water.",,139,0,0,[],0,[],False,False,NaN,False,
5,fgctirr,Nonmisogynistic,False,,"It's a question of the sales pitch involved.\n\nObviously, the people from Big Virgin had a better deal.",,102,0,0,[],0,[],False,False,NaN,False,
6,fgdomwf,Nonmisogynistic,False,,Some places have poor water quality. Virgin blood may have been less expensive than imported water.,,99,0,0,[],0,[],False,False,NaN,False,
7,fgbwoi5,Nonmisogynistic,False,,So if I drink enough water I turn into a baby?,,46,0,0,[],0,[],False,False,NaN,False,
8,fgbxtc0,Nonmisogynistic,False,,"You'll Benjamin Button yourself, yes.",,37,0,0,[],0,[],False,False,NaN,False,
9,fgdmluh,Nonmisogynistic,False,,Isn't this the plot of Cocoon?,,30,0,0,[],0,[],False,False,NaN,False,


## Missing Values, Duplicates, and Label Conflicts

In [19]:
duplicate_entry_ids = df.loc[df.duplicated("entry_id", keep=False), "entry_id"].nunique()
extra_rows_due_to_repeated_entry_id = len(df) - df["entry_id"].nunique()
exact_duplicate_rows = int(df.duplicated().sum())
duplicate_text_extra_rows = int(df.duplicated("text_for_audit").sum())
duplicate_text_groups = df.loc[df.duplicated("text_for_audit", keep=False), "text_for_audit"].nunique()
empty_text_rows = int(df["text_empty"].sum())
empty_text_posts = int(df.loc[df["text_empty"], "entry_id"].nunique())
conflicting_label_posts = post_summary.loc[post_summary["level_1_collapsed"].str.startswith("Conflict:")].index.tolist()

quality_summary = pd.DataFrame(
    [
        ("entry_id values appearing in multiple rows", duplicate_entry_ids),
        ("extra rows from repeated entry_id values", extra_rows_due_to_repeated_entry_id),
        ("exact duplicate full rows", exact_duplicate_rows),
        ("extra rows with duplicate repaired text", duplicate_text_extra_rows),
        ("duplicate repaired-text groups", duplicate_text_groups),
        ("rows with empty repaired text", empty_text_rows),
        ("unique posts with empty repaired text", empty_text_posts),
        ("rows with malformed image-column text fragments", int(df["malformed_image_value"].sum())),
        ("unique posts with malformed image-column text fragments", int(post_summary["has_malformed_image_value"].sum())),
        ("unique posts with conflicting level_1 labels", len(conflicting_label_posts)),
    ],
    columns=["metric", "value"],
)

display(quality_summary)

print("Rows with empty repaired text:")
display(df.loc[df["text_empty"], ["entry_id", "image", "week", "group", "sheet_order", "level_1", "split"]])

if conflicting_label_posts:
    print("Posts with conflicting level_1 labels:")
    display(df.loc[df["entry_id"].isin(conflicting_label_posts), ["entry_id", "text_for_audit", "level_1", "level_2", "level_3", "highlight"]])

,metric,value
0,entry_id values appearing in multiple rows,106
1,extra rows from repeated entry_id values,184
2,exact duplicate full rows,11
3,extra rows with duplicate repaired text,413
4,duplicate repaired-text groups,136
5,rows with empty repaired text,12
6,unique posts with empty repaired text,12
7,rows with malformed image-column text fragments,14
8,unique posts with malformed image-column text fragments,14
9,unique posts with conflicting level_1 labels,1


Rows with empty repaired text:


,entry_id,image,week,group,sheet_order,level_1,split
849,fikbg54,,3,1,"(15, 1, 1, 2, 1, 1)",Nonmisogynistic,train
944,fiaqwy6,,3,2,"(21, 1, 1)",Nonmisogynistic,train
946,fiaz4jx,,3,2,"(21, 2, 1)",Nonmisogynistic,train
1839,fk0yea3,,5,1,"(18, 1, 1)",Nonmisogynistic,train
2777,fkcvuy4,,6,2,"(53, 5)",Nonmisogynistic,train
2976,fmpkcuj,,7,1,"(15, 8)",Nonmisogynistic,train
4493,fp7a8sv,,9,2,"(60, 8)",Nonmisogynistic,train
4747,gail3k,Yes,9,2,"(89,)",Nonmisogynistic,train
4794,fprc2j9,,10,1,"(1, 4)",Nonmisogynistic,train
5433,fq08g44,,10,2,"(69, 1, 1, 1)",Nonmisogynistic,test


Posts with conflicting level_1 labels:


,entry_id,text_for_audit,level_1,level_2,level_3,highlight
6106,fr2197x,"1. Whatever vibration you put out, you will attract \n\nA lot of you sound like garbage (or just have very flawed, incel like, views/behavior) and wonder why you keep getting t...",Nonmisogynistic,Counter_speech,,"1. Whatever vibration you put out, you will attract \n\nA lot of you sound like garbage (or just have very flawed, incel like, views/behavior) and wonder why you keep getting t..."
6563,fr2197x,"1. Whatever vibration you put out, you will attract \n\nA lot of you sound like garbage (or just have very flawed, incel like, views/behavior) and wonder why you keep getting t...",Misogynistic,Misogynistic_pejorative,,sluts
6564,fr2197x,"1. Whatever vibration you put out, you will attract \n\nA lot of you sound like garbage (or just have very flawed, incel like, views/behavior) and wonder why you keep getting t...",Misogynistic,Misogynistic_pejorative,,bitches


## Answer Summary

In [20]:
readable_image_answer = (
    "not computed - EasyOCR did not run successfully"
    if readable_image_post_count is None
    else int(readable_image_post_count)
)

answers = pd.DataFrame(
    [
        ("How many total posts are there?", f"{df['entry_id'].nunique():,} unique posts ({len(df):,} final-label rows)"),
        ("How many are misogynistic vs non-misogynistic?", f"unique posts: {int((post_summary['level_1_collapsed'] == 'Misogynistic').sum()):,} misogynistic, {int((post_summary['level_1_collapsed'] == 'Nonmisogynistic').sum()):,} non-misogynistic, {int(post_summary['level_1_collapsed'].str.startswith('Conflict:').sum()):,} conflict; rows: {int((df['level_1'] == 'Misogynistic').sum()):,} misogynistic, {int((df['level_1'] == 'Nonmisogynistic').sum()):,} non-misogynistic"),
        ("How many posts have an associated image?", f"{int(post_summary['has_image'].sum()):,} unique posts ({int(df['image_flag'].sum()):,} rows marked image == Yes); {len(image_files):,} image files are present on disk"),
        ("How many posts contain Unicode emojis in the raw text?", f"{emoji_posts:,} unique posts ({emoji_rows:,} rows)"),
        ("How many image posts appear to contain readable text inside the image?", f"{readable_image_answer}; OCR status: {ocr_status}"),
        ("Where are OCR results cached?", str(OCR_RESULTS_CACHE)),
        ("What does each cached image OCR row include?", "image_id, file, ocr_text, ocr_success/ocr_status, confidence, processing_time_seconds, Unicode OCR emojis, OCR emoticons, and OCR laughter tokens"),
        ("Is the one-row-per-post dataframe available?", f"yes - `post_df` has {len(post_df):,} rows and the requested minimum columns"),
        ("Are there missing files, broken paths, duplicates, or empty texts?", f"missing expected image files: {len(missing_expected_images):,}; extra image files: {len(extra_image_files):,}; broken/unreadable image files: {int((~image_inspection_df['readable']).sum()) if len(image_inspection_df) else 0:,}; repeated-entry extra rows: {extra_rows_due_to_repeated_entry_id:,}; exact duplicate rows: {exact_duplicate_rows:,}; empty-text rows: {empty_text_rows:,}"),
    ],
    columns=["question", "answer"],
)

display(answers)


,question,answer
0,How many total posts are there?,"6,383 unique posts (6,567 final-label rows)"
1,How many are misogynistic vs non-misogynistic?,"unique posts: 515 misogynistic, 5,867 non-misogynistic, 1 conflict; rows: 699 misogynistic, 5,868 non-misogynistic"
2,How many posts have an associated image?,172 unique posts (173 rows marked image == Yes); 169 image files are present on disk
3,How many posts contain Unicode emojis in the raw text?,0 unique posts (0 rows)
4,How many image posts appear to contain readable text inside the image?,125; OCR status: loaded from OCR cache (169 cache hits); cache: /mnt/course-ee-559/rcp-caas-ee-559-g37/scratch-g37/EE559-DeepLearningProject-g37/.easyocr/ocr_results.csv
5,Where are OCR results cached?,/mnt/course-ee-559/rcp-caas-ee-559-g37/scratch-g37/EE559-DeepLearningProject-g37/.easyocr/ocr_results.csv
6,What does each cached image OCR row include?,"image_id, file, ocr_text, ocr_success/ocr_status, confidence, processing_time_seconds, Unicode OCR emojis, OCR emoticons, and OCR laughter tokens"
7,Is the one-row-per-post dataframe available?,"yes - `post_df` has 6,383 rows and the requested minimum columns"
8,"Are there missing files, broken paths, duplicates, or empty texts?",missing expected image files: 29; extra image files: 26; broken/unreadable image files: 0; repeated-entry extra rows: 184; exact duplicate rows: 11; empty-text rows: 12
